In [1]:
LON_MIN = 52.753105528788296
LON_MAX = 61.123379687612506
LAT_MIN = 22.106445540189902
LAT_MAX = 27.762549166787664

LAT_C = (LAT_MAX+LAT_MIN) / 2
LON_C = (LON_MAX+LON_MIN) / 2

In [78]:
import pandas as pd
import numpy as np

LOWER_T_VALUE = 6*2
LAT_CELLS = 300
LON_CELLS = 300

parquet = '../output/models/custom_ds_latent_size_128a_fivo_adamw/logprob.parquet'

_ds = pd.read_parquet(parquet, columns=['log_prob'])
_ds.index.levels[0]
lat_lon_ds = pd.read_parquet('../new_data/ais_test_by_bins.parquet', columns=['latitude', 'longitude'])
lat_lon_ds.index.levels[0]
_ds = pd.merge(_ds, lat_lon_ds, how='inner', left_index=True, right_index=True)
del lat_lon_ds
_ds = _ds.reset_index()
_ds = _ds.set_index('latitude')
_ds = _ds.set_index('longitude', append=True)
# higher than a timestamp
_ds = _ds[_ds['t']>= LOWER_T_VALUE]
_ds = _ds.drop(['t', 'track_id'], axis=1)
quantile=1.64
_ds = _ds[_ds.groupby(level=[0,1], sort=False)['log_prob'].transform(lambda ds: ( ds - ds.mean()) <= quantile * ds.std())]
_ds = _ds.groupby(level=[0,1]).agg(['mean', 'std', 'count'])


In [ ]:
_ds['lat'] = _ds.index.get_level_values(0) 
_ds['lon'] = _ds.index.get_level_values(1) 

In [117]:
_ds

log_prob                  COORDINATES  lat  lon
                         mean        std count                      
latitude longitude                                                  
0        259       -12.382470   5.176122    12           0    0  259
         281       -23.286079   8.433793     2           0    0  281
         292       -24.376383  19.629768     3           0    0  292
1        260       -32.812111  29.146357     2           1    1  260
         287        -8.008315   2.055020     2           1    1  287
...                       ...        ...   ...         ...  ...  ...
264      120       -45.647541   4.673352     4         264  264  120
         121       -44.765457   4.397018     2         264  264  121
         122       -43.336857   1.412104     2         264  264  122
         124       -44.266056   3.258330     3         264  264  124
         125       -41.279755   6.002734     2         264  264  125

[12228 rows x 6 columns]

In [ ]:

_ds['lat'] = LAT_MIN + _ds['lat'] * (LAT_MAX-LAT_MIN)/300
_ds['lon'] = LON_MIN + _ds['lon'] * (LON_MAX-LON_MIN)/300


log_prob                   \
                         mean        std count   
latitude longitude                               
0        259       -12.382470   5.176122    12   
         281       -23.286079   8.433793     2   
         292       -24.376383  19.629768     3   
1        260       -32.812111  29.146357     2   
         287        -8.008315   2.055020     2   
...                       ...        ...   ...   
264      120       -45.647541   4.673352     4   
         121       -44.765457   4.397018     2   
         122       -43.336857   1.412104     2   
         124       -44.266056   3.258330     3   
         125       -41.279755   6.002734     2   

                                                          COORDINATES  \
                                                                        
latitude longitude                                                      
0        259        [[22.106445540189902, 52.753105528788296], [26...   
         281        [[22.106445540189902, 52.753105528788296], [27...   
         292        [[22.106445540189902, 52.753105528788296], [27...   
1        260        [[22.12529921894523, 52.78100644265104], [27.0...   
         287        [[22.12529921894523, 52.78100644265104], [27.5...   
...                                                               ...   
264      120        [[27.08381673159593, 60.1189467885536], [24.36...   
         121        [[27.08381673159593, 60.1189467885536], [24.38...   
         122        [[27.08381673159593, 60.1189467885536], [24.40...   
         124        [[27.08381673159593, 60.1189467885536], [24.44...   
         125        [[27.08381673159593, 60.1189467885536], [24.46...   

                          lat        lon  
                                          
latitude longitude                        
0        259        22.106446  59.979442  
         281        22.106446  60.593262  
         292        22.106446  60.900172  
1        260        22.125299  60.007343  
         287        22.125299  60.760668  
...                       ...        ...  
264      120        27.083817  56.101215  
         121        27.083817  56.129116  
         122        27.083817  56.157017  
         124        27.083817  56.212819  
         125        27.083817  56.240720  

[12228 rows x 6 columns]

In [127]:
_ds['COORDINATES'] = list(map(list, zip(_ds['lat'], _ds['lon'])))

In [128]:
_ds

log_prob                   \
                         mean        std count   
latitude longitude                               
0        259       -12.382470   5.176122    12   
         281       -23.286079   8.433793     2   
         292       -24.376383  19.629768     3   
1        260       -32.812111  29.146357     2   
         287        -8.008315   2.055020     2   
...                       ...        ...   ...   
264      120       -45.647541   4.673352     4   
         121       -44.765457   4.397018     2   
         122       -43.336857   1.412104     2   
         124       -44.266056   3.258330     3   
         125       -41.279755   6.002734     2   

                                                 COORDINATES        lat  \
                                                                          
latitude longitude                                                        
0        259        [22.106445540189902, 59.979442219239864]  22.106446   
         281        [22.106445540189902, 60.593262324220305]  22.106446   
         292         [22.106445540189902, 60.90017237671053]  22.106446   
1        260          [22.12529921894523, 60.00734313310261]  22.125299   
         287         [22.12529921894523, 60.760667807396786]  22.125299   
...                                                      ...        ...   
264      120          [27.08381673159593, 56.10121519231798]  27.083817   
         121         [27.08381673159593, 56.129116106180724]  27.083817   
         122         [27.08381673159593, 56.157017020043476]  27.083817   
         124          [27.08381673159593, 56.21281884776897]  27.083817   
         125         [27.08381673159593, 56.240719761631716]  27.083817   

                          lon  
                               
latitude longitude             
0        259        59.979442  
         281        60.593262  
         292        60.900172  
1        260        60.007343  
         287        60.760668  
...                       ...  
264      120        56.101215  
         121        56.129116  
         122        56.157017  
         124        56.212819  
         125        56.240720  

[12228 rows x 6 columns]

In [133]:
df

,ADDRESS,RACKS,SPACES,COORDINATES
0,939 ELLIS ST,2,4,"[-122.42177834, 37.78346622]"
1,1380 HOWARD ST,1,2,"[-122.414411, 37.774458]"
2,1195 OAK ST,1,2,"[-122.438887, 37.772737]"
3,1387 VALENCIA ST,1,2,"[-122.42019976, 37.75087429]"
4,180 TOWNSEND ST,1,2,"[-122.392606, 37.779369]"
...,...,...,...,...
2515,900 VALENCIA ST,1,2,"[-122.42152025, 37.75825474]"
2516,91 WALTER ST,1,2,"[-122.432136, 37.767817]"
2517,916 KEARNY,1,2,"[-122.40502124, 37.79653486]"
2518,939 EDDY ST,1,2,"[-122.423112, 37.782316]"


In [134]:
_ds.reset_index()[['COORDINATES']]

,COORDINATES
,
0,"[22.106445540189902, 59.979442219239864]"
1,"[22.106445540189902, 60.593262324220305]"
2,"[22.106445540189902, 60.90017237671053]"
3,"[22.12529921894523, 60.00734313310261]"
4,"[22.12529921894523, 60.760667807396786]"
...,...
12223,"[27.08381673159593, 56.10121519231798]"
12224,"[27.08381673159593, 56.129116106180724]"
12225,"[27.08381673159593, 56.157017020043476]"


In [132]:
import pydeck as pdk
import pandas as pd

CPU_GRID_LAYER_DATA = (
    "https://raw.githubusercontent.com/uber-common/" "deck.gl-data/master/website/sf-bike-parking.json"
)
df = pd.read_json(CPU_GRID_LAYER_DATA)

# Define a layer to display on a map

layer = pdk.Layer(
    "GridLayer",
    _ds.reset_index()[['COORDINATES']],
    pickable=True,
    extruded=True,
    cell_size=200,
    elevation_scale=4,
    get_position="COORDINATES",
)

view_state = pdk.ViewState(latitude=LAT_C, longitude=LON_C, zoom=7, bearing=0, pitch=45)

# Render
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
)
r

TypeError: keys must be str, int, float, bool or None, not tuple

TypeError: keys must be str, int, float, bool or None, not tuple